# Ultravox v0.5 (Llama-3.2-1B)

In [ ]:
import os
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import transformers
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
!source /etc/network_turbo

In [ ]:
CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "fixie-ai/ultravox-v0_5-llama-3_2-1b"

os.environ["HF_HOME"] = CACHE_DIR

pipe = transformers.pipeline(
    model=MODEL_ID,
    trust_remote_code=True,
    model_kwargs={"cache_dir": CACHE_DIR},
)

print("Ultravox model loaded.")

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [ ]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="ultravox_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [ ]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    audio, sr = librosa.load(str(wav_path), sr=16000, mono=True)

    turns = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"<|audio|>\n{USER_PROMPT}"},
    ]

    output = pipe(
        {"audio": audio, "turns": turns, "sampling_rate": sr},
        max_new_tokens=64,
    )
    return output

In [ ]:
OUTPUT_DIR = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/LLM/ultravox_result")


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            pred = parse_prediction(raw)
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [ ]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt_origin", "Pitt_origin", "Pitt_origin_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt_origin"
evaluate_dataset(csv, audio_dir, "Pitt-origin-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Denoiser")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-origin-FRCRN_SE")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-origin-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Resemble")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")